# Hadoop hands-on: from standalone to a pseudo-cluster (Google Colab)

In this session you will **install and run Hadoop yourself**, step by step, on the Ubuntu machine that Colab gives you.

| Part | What you will do |
|---|---|
| 1 | Prepare the machine: Java, environment variables, download Hadoop **3.5.0** |
| 2 | **Standalone mode**: one Java process, local files, no daemons |
| 3 | **Pseudo-distributed mode**: HDFS + YARN daemons on one machine |
| 4 | 🧩 **Final exercise**: every student solves a different Sudoku with Hadoop |

**How it works.** Every step has a *goal* and the *expected result*. The code cells are **empty: you write the commands**.
Open the 💡 *Hint* if you get stuck. Run the cells **in order**.

> ⚠️ A Colab runtime is **temporary**: if it is recycled you lose everything. Re-run the notebook from the top
> (or use *Runtime → Disconnect and delete runtime* to start from a clean machine).
> Colab cells run **as root** and there is **no `systemd`**: that is why some steps differ from a normal Ubuntu VM.

# Part 1 – Preparing the machine

## 1.1 Java 17

Hadoop is written in Java. **Hadoop 3.5.0 requires Java 17** on the server side. Install the OpenJDK 17 **headless JDK** with `apt` and check the version.

*Note: Colab already ships with Java 21 as default, so after installing 17 you will have both. We will tell Hadoop which one to use in the next step.*

<details><summary>💡 Hint</summary>

Update the package index first. The package is called `openjdk-17-jdk-headless`. Use `-y` so `apt` does not ask questions.
Start the cell with `%%bash` to run several shell lines in one cell, and `set -e` to stop on the first error.

</details>

✅ **Expected:** The path `/usr/lib/jvm/java-17-openjdk-amd64` exists and its `java -version` says 17.

In [1]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
ls -d /usr/lib/jvm/java-17*
/usr/lib/jvm/java-17-openjdk-amd64/bin/java -version

/usr/lib/jvm/java-17-openjdk-amd64


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.21.1" 2026-11-17
OpenJDK Runtime Environment (build 17.0.21.1+2-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.1+2-1-24.04.1-Ubuntu, mixed mode, sharing)


## 1.2 Environment variables

Hadoop needs to know where Java and Hadoop itself live. On a normal VM you would add `export` lines to `~/.bashrc`.
In Colab we set them **from Python** (`os.environ`) so every following cell inherits them.

Define: `JAVA_HOME` (Java 17), `HADOOP_HOME` (`/usr/local/hadoop`, where we will install it), `HADOOP_CONF_DIR`, and add `bin`/`sbin` of Hadoop and Java's `bin` to the `PATH`.

<details><summary>💡 Hint</summary>

Python cell. `import os`, then `os.environ["NAME"] = "value"`.
Hadoop also likes `HADOOP_MAPRED_HOME`, `HADOOP_COMMON_HOME`, `HADOOP_HDFS_HOME` and `YARN_HOME` pointing to `HADOOP_HOME`.
`HADOOP_CONF_DIR` is `$HADOOP_HOME/etc/hadoop`. Put the new directories **before** the old `PATH` so Java 17 wins over Java 21.

</details>

✅ **Expected:** Printing `JAVA_HOME` shows the Java 17 path. *(If Colab restarts the runtime you must run this cell again.)*

In [3]:
import os
HADOOP_VERSION = "3.5.0"
HADOOP_HOME = "/usr/local/hadoop"
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ.update({
 "HADOOP_VERSION": HADOOP_VERSION,
 "JAVA_HOME": JAVA_HOME,
 "HADOOP_HOME": HADOOP_HOME,
 "HADOOP_MAPRED_HOME": HADOOP_HOME,
 "HADOOP_COMMON_HOME": HADOOP_HOME,
 "HADOOP_HDFS_HOME": HADOOP_HOME,
 "YARN_HOME": HADOOP_HOME,
 "HADOOP_CONF_DIR": HADOOP_HOME + "/etc/hadoop",
})
os.environ["PATH"] = f"{JAVA_HOME}/bin:{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64


## 1.3 Download and unpack Hadoop

Download the binary release **hadoop-3.5.0.tar.gz** from the Apache download server, unpack it and move it to `/usr/local/hadoop`.
Finally run `hadoop version` to check that the installation works.

URL: `https://dlcdn.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz` (≈ 550 MB). If it is not there any more (old releases are removed), use the same path on `https://archive.apache.org/dist/` (much slower).

<details><summary>💡 Hint</summary>

`wget -q <url>`, `tar -xzf <file>`, `mv <folder> /usr/local/hadoop`. Work in `/content`.
Make the cell *re-runnable*: only download if `/usr/local/hadoop` does not exist yet (`if [ ! -d ... ]; then ... fi`).

</details>

✅ **Expected:** `hadoop version` prints `Hadoop 3.5.0`.

In [5]:
%%bash
set -e
if [ ! -d /usr/local/hadoop ]; then
 cd /content
 F=hadoop-$HADOOP_VERSION/hadoop-$HADOOP_VERSION.tar.gz
 wget -q https://dlcdn.apache.org/hadoop/common/$F || wget -q https://archive.apache.org/dist/hadoop/common/$F
 tar -xzf hadoop-$HADOOP_VERSION.tar.gz
 mv hadoop-$HADOOP_VERSION /usr/local/hadoop
fi
hadoop version


Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /usr/local/hadoop/share/hadoop/common/hadoop-common-3.5.0.jar


## 1.4 What is inside?

List the content of the Hadoop folder and of `share/hadoop/mapreduce`. Which folder has the configuration? Where are the commands? Where is the **examples jar**?

<details><summary>💡 Hint</summary>

Just `ls`.

</details>

✅ **Expected:** You see `bin`, `sbin`, `etc`, `share`… and a file named `hadoop-mapreduce-examples-3.5.0.jar`.

In [6]:
!ls /usr/local/hadoop
!ls /usr/local/hadoop/share/hadoop/mapreduce | head


bin	    etc      libexec	      LICENSE.txt    README.txt
compose     include  LICENSE-binary   NOTICE-binary  sbin
Dockerfile  lib      licenses-binary  NOTICE.txt     share
hadoop-mapreduce-client-app-3.5.0.jar
hadoop-mapreduce-client-common-3.5.0.jar
hadoop-mapreduce-client-core-3.5.0.jar
hadoop-mapreduce-client-hs-3.5.0.jar
hadoop-mapreduce-client-hs-plugins-3.5.0.jar
hadoop-mapreduce-client-jobclient-3.5.0.jar
hadoop-mapreduce-client-jobclient-3.5.0-tests.jar
hadoop-mapreduce-client-nativetask-3.5.0.jar
hadoop-mapreduce-client-shuffle-3.5.0.jar
hadoop-mapreduce-client-uploader-3.5.0.jar


# Part 2 – Standalone mode

Standalone is Hadoop's default mode: **a single Java process**, no HDFS and no YARN, reading and writing the **local** file system.
It is ideal to test that the installation works and to debug MapReduce code.

## 2.1 Your first MapReduce job: `grep`

Use the examples jar to **count how many times `<configuration>` and `</configuration>` appear** in Hadoop's own XML configuration files.

1. Create a folder `input` inside `/usr/local/hadoop` and copy there all the `*.xml` files of `etc/hadoop`.
2. Run the `grep` example: `hadoop jar <examples jar> grep <input> <output> '<regular expression>'`.
3. Show the result.

The regular expression `'<[/]*configuration>'` matches both tags.

<details><summary>💡 Hint</summary>

⚠️ The **output folder must not exist**, otherwise the job fails: remove it first with `rm -rf`.
The job prints only a summary (counters). **The result is in the output folder**: `cat` its files.
Use `%%bash`, `cd /usr/local/hadoop` and pipe the job output through `tail -n 25` to keep the cell short.

</details>

✅ **Expected:** The result shows `10 <configuration>` and `10 </configuration>`.

In [9]:
%%bash
cd /usr/local/hadoop
rm -rf input grep_example
mkdir input
cp etc/hadoop/*.xml input/
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.5.0.jar grep ./input ./grep_example '<[/]*configuration>' 2>&1 | tail -n 15
echo "----- RESULT -----"
cat ./grep_example/*

		Failed Shuffles=0
		Merged Map outputs=1
		GC time elapsed (ms)=0
		Total committed heap usage (bytes)=1220542464
	Shuffle Errors
		BAD_ID=0
		CONNECTION=0
		IO_ERROR=0
		WRONG_LENGTH=0
		WRONG_MAP=0
		WRONG_REDUCE=0
	File Input Format Counters 
		Bytes Read=163
	File Output Format Counters 
		Bytes Written=51
----- RESULT -----
10	<configuration>
10	</configuration>


## 2.2 Estimating π

Run the `pi` example (quasi-Monte Carlo method) with **16 maps** and **1,000,000 samples per map**.

<details><summary>💡 Hint</summary>

`hadoop jar <examples jar> pi <maps> <samples>`.

</details>

✅ **Expected:** A line like `Estimated value of Pi is 3.14159…`.

In [10]:
%%bash
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.5.0.jar pi 16 1000000 2>&1 | tail -n 3


		Bytes Written=109
Job Finished in 5.792 seconds
Estimated value of Pi is 3.14159125000000000000


❓ **Question:** Why does the estimate improve with more samples? Which part is the "map" and which the "reduce"?

✍ *Your answer*:
- **Why the estimate improves:** According to the Law of Large Numbers in the quasi-Monte Carlo method, as the number of random samples (points) increases, the empirical ratio of points falling inside the unit circle to total points converges to the actual mathematical ratio (which is $\pi/4$). Thus, more samples reduce the statistical variance of the estimate.
- **Map part:** Each Map task independently generates a subset of random points (samples) and counts how many of those points fall inside the unit circle.
- **Reduce part:** The Reduce task aggregates (sums up) the counts of internal points and total points from all mappers, and then computes the final estimation of $\pi$ using the formula: $\pi \approx 4 \times \frac{\text{total points inside}}{\text{total points generated}}$.

## 2.3 What other examples are there?

Run the examples jar **without any program name** to get the list of available examples. Read it carefully: one of them (`sudoku`) will be your final exercise.

<details><summary>💡 Hint</summary>

Same `hadoop jar <examples jar>` but with nothing after it. It prints a usage message (that is fine).

</details>

✅ **Expected:** A list of ~20 programs (`aggregatewordcount`, `pi`, `sudoku`, `terasort`, `wordcount`…).

In [12]:
%%bash
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.5.0.jar 2>&1 | head -n 40 || true

An example program must be given as the first argument.
Valid program names are:
  aggregatewordcount: An Aggregate based map/reduce program that counts the words in the input files.
  aggregatewordhist: An Aggregate based map/reduce program that computes the histogram of the words in the input files.
  bbp: A map/reduce program that uses Bailey-Borwein-Plouffe to compute exact digits of Pi.
  dbcount: An example job that count the pageview counts from a database.
  distbbp: A map/reduce program that uses a BBP-type formula to compute exact bits of Pi.
  grep: A map/reduce program that counts the matches of a regex in the input.
  join: A job that effects a join over sorted, equally partitioned datasets
  multifilewc: A job that counts words from several files.
  pentomino: A map/reduce tile laying program to find solutions to pentomino problems.
  pi: A map/reduce program that estimates Pi using a quasi-Monte Carlo method.
  randomtextwriter: A map/reduce program that writes 10GB of r

# Part 3 – Pseudo-distributed mode

Now Hadoop runs as **separate daemons** (each one a JVM process talking over the network) on a single machine:

| HDFS (storage) | YARN (resources) |
|---|---|
| `NameNode` – metadata | `ResourceManager` – master |
| `DataNode` – stores blocks | `NodeManager` – worker |
| `SecondaryNameNode` – checkpoints | |

The start scripts launch daemons **over SSH**, so we first need a passwordless `ssh localhost`.

## 3.1 Passwordless SSH to localhost

1. Install the SSH server (`openssh-server`).
2. **Start the `sshd` daemon.**
3. Create an RSA key pair **without passphrase**, authorize it for yourself and make `ssh localhost` work **without any question**.
4. Test it with `ssh localhost "echo hello from $(whoami)"`.

⚠️ Colab specifics: there is no `systemd`, and Colab's default `/etc/ssh/sshd_config` sets `Port 2222`, so `service ssh start` would **not** listen on port 22.

<details><summary>💡 Hint</summary>

- `mkdir -p /run/sshd`, then start the daemon yourself: `/usr/sbin/sshd -p 22 -o ListenAddress=127.0.0.1` (options on the command line override the config file). Wait 2 s.
- Keys: `ssh-keygen -t rsa -P "" -f ~/.ssh/id_rsa`, then append `~/.ssh/id_rsa.pub` to `~/.ssh/authorized_keys` (permissions `0600`).
- To avoid the *"are you sure you want to continue connecting?"* question, write `~/.ssh/config` with `StrictHostKeyChecking no`, `UserKnownHostsFile /dev/null` and `LogLevel ERROR` under `Host *`.

</details>

✅ **Expected:** `SSH to localhost works as root`, with no password and no questions.

In [14]:
%%bash
set -e
apt-get -qq install -y openssh-server > /dev/null
mkdir -p /run/sshd
pgrep -f "sshd -p 22" > /dev/null || /usr/sbin/sshd -p 22 -o ListenAddress=127.0.0.1
sleep 2

mkdir -p ~/.ssh && chmod 700 ~/.ssh
[ -f ~/.ssh/id_rsa ] || ssh-keygen -t rsa -P "" -f ~/.ssh/id_rsa -q
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 0600 ~/.ssh/authorized_keys
printf "Host *\n StrictHostKeyChecking no\n UserKnownHostsFile /dev/null\n LogLevel ERROR\n" > ~/.ssh/config
chmod 600 ~/.ssh/config
ssh localhost "echo SSH to localhost works as $(whoami)"

SSH to localhost works as root


### Configuration files (steps 3.2 – 3.6)
All of them live in `/usr/local/hadoop/etc/hadoop`. In a normal VM you would edit them with `nano`; here you will **create the files from a cell**
(`%%writefile <path>` writes the cell content into a file). Each XML file has the same shape:

```xml
<?xml version="1.0"?>
<configuration>
  <property><name>PROPERTY</name><value>VALUE</value></property>
</configuration>
```

## 3.2 `hadoop-env.sh`

Daemons are started through SSH, so they **do not inherit** our Python variables. Append these lines to `hadoop-env.sh`:

| Variable | Value |
|---|---|
| `JAVA_HOME` | the Java 17 path |
| `HADOOP_HOME` | `/usr/local/hadoop` |
| `HDFS_NAMENODE_USER`, `HDFS_DATANODE_USER`, `HDFS_SECONDARYNAMENODE_USER`, `YARN_RESOURCEMANAGER_USER`, `YARN_NODEMANAGER_USER` | `root` (Colab runs everything as root and Hadoop refuses unless you say it is intended) |

⚠️ Run this cell **only once** (it *appends*).

<details><summary>💡 Hint</summary>

`cat >> /usr/local/hadoop/etc/hadoop/hadoop-env.sh <<EOF … EOF` inside a `%%bash` cell, with `export NAME=value` lines.

</details>

✅ **Expected:** The `tail` of the file shows your 7 `export` lines.

In [15]:
%%bash
cat >> /usr/local/hadoop/etc/hadoop/hadoop-env.sh <<EOF
export JAVA_HOME=$JAVA_HOME
export HADOOP_HOME=$HADOOP_HOME
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
export YARN_RESOURCEMANAGER_USER=root
export YARN_NODEMANAGER_USER=root
EOF
tail -n 8 /usr/local/hadoop/etc/hadoop/hadoop-env.sh

# export HADOOP_REGISTRYDNS_SECURE_EXTRA_OPTS="-jvm server"
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export HADOOP_HOME=/usr/local/hadoop
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
export YARN_RESOURCEMANAGER_USER=root
export YARN_NODEMANAGER_USER=root


## 3.3 `core-site.xml`

Set the **default file system** to HDFS running on this machine.

| name | value |
|---|---|
| `fs.defaultFS` | `hdfs://localhost:9000` |

<details><summary>💡 Hint</summary>

`%%writefile /usr/local/hadoop/etc/hadoop/core-site.xml` and the XML template above.

</details>

✅ **Expected:** “Writing /usr/local/hadoop/etc/hadoop/core-site.xml”.

In [16]:
%%writefile /usr/local/hadoop/etc/hadoop/core-site.xml
<?xml version="1.0"?>
<configuration>
 <property>
 <name>fs.defaultFS</name>
 <value>hdfs://localhost:9000</value>
 </property>
</configuration>

Overwriting /usr/local/hadoop/etc/hadoop/core-site.xml


## 3.4 `hdfs-site.xml`

Configure HDFS.

| name | value | why |
|---|---|---|
| `dfs.replication` | `1` | we only have **one** DataNode |
| `dfs.namenode.secondary.http-address` | `localhost:9868` | so the start script reaches the SecondaryNameNode through `localhost` (our `sshd` only listens there) |

<details><summary>💡 Hint</summary>

Same as before, with **two** `<property>` entries.

</details>

✅ **Expected:** File written.

In [17]:
%%writefile /usr/local/hadoop/etc/hadoop/hdfs-site.xml
<?xml version="1.0"?>
<configuration>
 <property>
 <name>dfs.replication</name>
 <value>1</value>
 </property>
 <property>
 <name>dfs.namenode.secondary.http-address</name>
 <value>localhost:9868</value>
 </property>
</configuration>

Overwriting /usr/local/hadoop/etc/hadoop/hdfs-site.xml


## 3.5 `mapred-site.xml`

Tell MapReduce to run **on YARN**.

| name | value |
|---|---|
| `mapreduce.framework.name` | `yarn` |
| `yarn.app.mapreduce.am.env` | `HADOOP_MAPRED_HOME=/usr/local/hadoop` |
| `mapreduce.map.env` | `HADOOP_MAPRED_HOME=/usr/local/hadoop` |
| `mapreduce.reduce.env` | `HADOOP_MAPRED_HOME=/usr/local/hadoop` |

<details><summary>💡 Hint</summary>

Four `<property>` entries.

</details>

✅ **Expected:** File written.

In [18]:
%%writefile /usr/local/hadoop/etc/hadoop/mapred-site.xml
<?xml version="1.0"?>
<configuration>
 <property>
 <name>mapreduce.framework.name</name>
 <value>yarn</value>
 </property>
 <property>
 <name>yarn.app.mapreduce.am.env</name>
 <value>HADOOP_MAPRED_HOME=/usr/local/hadoop</value>
 </property>
 <property>
 <name>mapreduce.map.env</name>
 <value>HADOOP_MAPRED_HOME=/usr/local/hadoop</value>
 </property>
 <property>
 <name>mapreduce.reduce.env</name>
 <value>HADOOP_MAPRED_HOME=/usr/local/hadoop</value>
 </property>
</configuration>

Overwriting /usr/local/hadoop/etc/hadoop/mapred-site.xml


❓ **Question:** What would happen to the jobs if we did not set the first property?

✍ *Your answer*:
If we do not set `mapreduce.framework.name` to `yarn`, MapReduce would fall back to its default value, which is `local` (standalone mode). Consequently, jobs would run locally as a single Java thread on the client machine instead of being submitted to and distributed across the YARN cluster's ResourceManager and NodeManagers.

## 3.6 `yarn-site.xml`

Configure the **shuffle service** that moves map output to the reducers.

| name | value |
|---|---|
| `yarn.nodemanager.aux-services` | `mapreduce_shuffle` |
| `yarn.nodemanager.aux-services.mapreduce.shuffle.class` | `org.apache.hadoop.mapred.ShuffleHandler` |

<details><summary>💡 Hint</summary>

Two `<property>` entries.

</details>

✅ **Expected:** File written.

In [19]:
%%writefile /usr/local/hadoop/etc/hadoop/yarn-site.xml
<?xml version="1.0"?>
<configuration>
 <property>
 <name>yarn.nodemanager.aux-services</name>
 <value>mapreduce_shuffle</value>
 </property>
 <property>
 <name>yarn.nodemanager.aux-services.mapreduce.shuffle.class</name>
 <value>org.apache.hadoop.mapred.ShuffleHandler</value>
 </property>
</configuration>

Overwriting /usr/local/hadoop/etc/hadoop/yarn-site.xml


## 3.7 Format the NameNode

Initialise the HDFS metadata (`hdfs namenode -format`). ⚠️ Do it **only once**: formatting again wipes the file system. Use `-force` so it does not ask for confirmation, and show only the last lines.

<details><summary>💡 Hint</summary>

`hdfs namenode -format -force 2>&1 | tail -n 5`

</details>

✅ **Expected:** The tail ends with `SHUTDOWN_MSG: Shutting down NameNode …` (a successful format also prints *“has been successfully formatted”* above).

In [20]:
!hdfs namenode -format -force 2>&1 | tail -n 5


2026-11-24 15:06:24,889 INFO namenode.FSImage: FSImageSaver clean checkpoint: txid=0 when meet shutdown.
2026-11-24 15:06:24,890 INFO namenode.NameNode: SHUTDOWN_MSG: 
/************************************************************
SHUTDOWN_MSG: Shutting down NameNode at e37b54a73de5/172.28.0.12
************************************************************/


## 3.8 Start HDFS

Start the HDFS daemons with the script `start-dfs.sh` and list the Java processes with `jps`.

<details><summary>💡 Hint</summary>

The scripts are in `$HADOOP_HOME/sbin`, which is already in your `PATH`. Wait ~10 s before `jps`.

</details>

✅ **Expected:** `jps` shows **NameNode, DataNode and SecondaryNameNode**.  🔧 If one is missing, read its log: `tail -n 50 /usr/local/hadoop/logs/*datanode*.log`.

In [21]:
%%bash
start-dfs.sh
sleep 10
jps

Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [localhost]
9034 NameNode
9132 DataNode
9628 Jps
9327 SecondaryNameNode


## 3.9 Start YARN

Start YARN with `start-yarn.sh` and check the processes again.

<details><summary>💡 Hint</summary>

Same idea as 3.8.

</details>

✅ **Expected:** Two new processes: **ResourceManager and NodeManager** → 🎉 you have a pseudo-distributed Hadoop.

In [22]:
%%bash
start-yarn.sh
sleep 10
jps

Starting resourcemanager
Starting nodemanagers
10224 ResourceManager
10565 Jps
10328 NodeManager
9034 NameNode
9132 DataNode
9327 SecondaryNameNode


## 3.10 The same `grep`, now on HDFS

Repeat the `grep` job, but this time the input and output live **in HDFS**:

1. Create `/input` in HDFS and copy the XML files of `etc/hadoop` into it (`hadoop fs -mkdir`, `-copyFromLocal`).
2. Run the `grep` example reading `/input` and writing `/grep_example`.
3. Show the result with `hadoop fs -cat`.

⏱️ On Colab's 2 vCPUs YARN needs **a few minutes** to start the containers (2-4 min). That is normal.

<details><summary>💡 Hint</summary>

Make the cell re-runnable by removing `/input` and `/grep_example` first (`hadoop fs -rm -r -f`).

</details>

✅ **Expected:** Same result as in standalone: `10 <configuration>` and `10 </configuration>`.

In [23]:
%%bash
cd $HADOOP_HOME
hadoop fs -rm -r -f /input /grep_example
hadoop fs -mkdir /input
hadoop fs -copyFromLocal etc/hadoop/*.xml /input
hadoop fs -ls /input
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.5.0.jar grep /input /grep_example '<[/]*configuration>' 2>&1 | tail -n 5
echo "----- RESULT -----"
hadoop fs -cat /grep_example/*

Found 10 items
-rw-r--r--   1 root supergroup       9213 2026-11-24 15:11 /input/capacity-scheduler.xml
-rw-r--r--   1 root supergroup        145 2026-11-24 15:11 /input/core-site.xml
-rw-r--r--   1 root supergroup      14007 2026-11-24 15:11 /input/hadoop-policy.xml
-rw-r--r--   1 root supergroup        683 2026-11-24 15:11 /input/hdfs-rbf-site.xml
-rw-r--r--   1 root supergroup        234 2026-11-24 15:11 /input/hdfs-site.xml
-rw-r--r--   1 root supergroup        620 2026-11-24 15:11 /input/httpfs-site.xml
-rw-r--r--   1 root supergroup       3518 2026-11-24 15:11 /input/kms-acls.xml
-rw-r--r--   1 root supergroup        682 2026-11-24 15:11 /input/kms-site.xml
-rw-r--r--   1 root supergroup        481 2026-11-24 15:11 /input/mapred-site.xml
-rw-r--r--   1 root supergroup        307 2026-11-24 15:11 /input/yarn-site.xml
		WRONG_REDUCE=0
	File Input Format Counters 
		Bytes Read=151
	File Output Format Counters 
		Bytes Written=39
----- RESULT -----
10	<configuration>
10	</configurati

❓ **Question:** Compare with 2.1: what changed in the paths? Where is the data physically stored now?

✍ *Your answer*:
- **What changed in the paths:** In step 2.1, the paths (`./input` and `./grep_example`) pointed to directories on the local machine's ext4 file system. In step 3.10, the paths (`/input` and `/grep_example`) refer to paths inside the HDFS namespace.
- **Where data is physically stored:** The data is physically stored inside the block directory managed by the HDFS DataNode (located by default under `/tmp/hadoop-root/dfs/data/current/BP-.../blk_...` on the local file system of the machine hosting the DataNode daemon).

## 3.11 Inspect the cluster

In a VM you would open the web UIs (NameNode `:9870`, ResourceManager `:8088`). In Colab those pages **cannot be embedded** in the notebook, so first ask the command line (in the next steps you will also open the real pages in your own browser with *ngrok*):

1. `hdfs dfsadmin -report` – capacity and DataNodes.
2. `hdfs fsck /grep_example -files -blocks` – files and blocks of your output.
3. `yarn node -list` and `yarn application -list -appStates FINISHED` – workers and the jobs you have run.

<details><summary>💡 Hint</summary>

Use `head` on the first one, it is long.

</details>

✅ **Expected:** 1 live DataNode, `replication=1` for your files, 1 NodeManager `RUNNING` and your finished applications.

In [24]:
%%bash
hdfs dfsadmin -report | head -n 12
echo "-----"
hdfs fsck /grep_example -files -blocks | head -n 10
echo "-----"
yarn node -list
yarn application -list -appStates FINISHED

Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 112708493312 (104.97 GB)
DFS Remaining: 112707596288 (104.97 GB)
DFS Used: 897024 (876 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Badly Distributed Blocks: 0
	Low redundancy blocks with highest priority to recover: 0
-----
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path /grep_example at Tue Nov 24 15:16:39 UTC 2026

/grep_example <dir>
/grep_example/_SUCCESS 0 bytes, replicated: replication=1, 0 block(s):  OK

/grep_example/part-r-00000 39 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-157404741-172.28.0.12-1795532784158:blk_1073741851_1027 len=39 Live_repl=1


Status: HEALTHY
-----
Total Nodes:1
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
e37b54a73de5:46181	        RUNNING	e37b54a73de5:8042	                           0
Total number of applicat

Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&path=%2Fgrep_example
2026-11-24 15:16:42,259 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-11-24 15:16:45,731 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032


## 3.12 Install ngrok and create your account

Colab machines have no public address, so your browser cannot reach `localhost:9870`. **ngrok** creates a temporary public URL that forwards to a port of the Colab machine.

1. Create a **free account** at <https://dashboard.ngrok.com/signup> and copy your **authtoken** (*Your Authtoken* in the dashboard). It is personal: **do not share it and do not leave it written in a notebook you hand in or publish**.
2. Install the ngrok agent with `apt` (add ngrok's apt repository and key first) and register your token with `ngrok config add-authtoken`.
3. Check it with `ngrok version`.

<details><summary>💡 Hint</summary>

Repository: download the key from `https://ngrok-agent.s3.amazonaws.com/ngrok.asc` into `/etc/apt/trusted.gpg.d/ngrok.asc`, then write the line `deb https://ngrok-agent.s3.amazonaws.com buster main` into `/etc/apt/sources.list.d/ngrok.list`. Then `apt-get update` and `apt-get install -y ngrok`.
Replace `PASTE_YOUR_TOKEN_HERE` by your token only while you run the cell, and **delete it afterwards**.

</details>

✅ **Expected:** `ngrok version ...` and `Authtoken saved to configuration file`.

In [33]:
%%bash
curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc >/etc/apt/trusted.gpg.d/ngrok.asc
echo "deb https://ngrok-agent.s3.amazonaws.com buster main" >/etc/apt/sources.list.d/ngrok.list
apt-get -qq update
apt-get -qq install -y ngrok > /dev/null
ngrok config add-authtoken 3MYiSNYzASOiM5WNBsyMoPfdDMO_7bbabzCktJx9JHnLhEqDw
ngrok version

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
ngrok version 3.39.13


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 3.13 Open the HDFS and YARN web pages

Open **two tunnels at once** (the free plan allows it): one to the **NameNode** page (port `9870`) and one to the **ResourceManager** page (port `8088`).

1. Write a small ngrok configuration file `/content/tunnels.yml` (`%%writefile`) that defines the two tunnels, `hdfs` and `yarn`.
2. Start ngrok in the **background** with both tunnels (the cell must not block).
3. Ask ngrok's local API (`http://127.0.0.1:4040/api/tunnels`) for the public URLs and print them.
4. Open both URLs in your browser. If ngrok shows a warning page, click **Visit Site**.

<details><summary>💡 Hint</summary>

Config (version 2 syntax):
```
version: "2"
tunnels:
  hdfs:
    proto: http
    addr: 9870
  yarn:
    proto: http
    addr: 8088
```
Start it with `nohup ngrok start --all --config /root/.config/ngrok/ngrok.yml,/content/tunnels.yml > /content/ngrok.log 2>&1 &` (the first file holds your token). Wait a few seconds before asking the API; each tunnel has a `public_url` field.

</details>

✅ **Expected:** Two `https://....ngrok-free.app` URLs. The first shows the NameNode (*Overview*, *Datanodes*, *Utilities > Browse the file system*); the second shows YARN with your finished `grep` applications.

In [38]:
%%writefile /content/tunnels.yml
version: "2"
tunnels:
  hdfs:
    proto: http
    addr: 9870
  yarn:
    proto: http
    addr: 8088

Overwriting /content/tunnels.yml


In [39]:
%%bash
pkill ngrok || true
nohup ngrok start --all --config /root/.config/ngrok/ngrok.yml,/content/tunnels.yml > /content/ngrok.log 2>&1 &
sleep 6
if ! curl -s http://127.0.0.1:4040/api/tunnels | python3 -c "import sys,json; [print(t['name'], t['public_url']) for t in json.load(sys.stdin)['tunnels']]"; then
  echo "=== NGROK LOGS ==="
  cat /content/ngrok.log
fi

hdfs https://authentic-yoyo-luckiness.ngrok-free.dev
yarn https://authentic-yoyo-luckiness.ngrok-free.dev


❓ **Question:** In the NameNode page, open *Utilities > Browse the file system*: do you see your `grep_example` folder? In the YARN page, how many applications have finished and what is their final status? (Links inside these pages may point to the internal hostname and not work: that is normal.)

✍ *Your answer*:
- **Browse File System:** Yes, the `/grep_example` folder is visible in the HDFS directory tree.
- **YARN Applications:** There are exactly 2 finished applications (`grep-search` and `grep-sort`), both displaying a final status of `SUCCEEDED`.

# Part 4 – 🧩 Final exercise: solve your Sudoku

Hadoop's examples jar includes a **Sudoku solver** (it uses the *dancing links* exact-cover algorithm).
Your teacher has published **a different puzzle for each student** in a GitHub repository, one text file per student. Your mission: download yours into Colab and let Hadoop solve it.

The file has 9 lines of 9 symbols separated by spaces, with `?` for the empty cells:

```
8 5 ? 3 9 ? ? ? ?
? ? 2 ? ? ? ? ? ?
...
```

## 4.1 Download your Sudoku

Your puzzle file is called `sudoku_<CODE>.txt`, where **CODE = the first 2 + the last 4 characters of your ID number**, in capital letters (6 characters). For example, the ID `12345678Z` gives the code `12678Z`.

The files are in the teacher's GitHub repository:

`https://raw.githubusercontent.com/AntSanVar/DistributedSystemData/main/Session01/sudoku_<CODE>.txt`

Download **your** file into `/content/sudoku.txt` with `wget`, print it, and count how many numbers it has (the *clues*; the rest are `?`).

*(Alternative if the download fails: download the file to your computer and upload it with the **Files** panel 📁 on the left bar → *Upload to session storage*; name it `sudoku.txt`.)*

<details><summary>💡 Hint</summary>

Put your code in a shell variable (`CODE=12678Z`) and use it in the URL. `wget -q -O /content/sudoku.txt <url>` saves it with the name `sudoku.txt`. If you get a *404* error, your code is wrong: check the rule above. To count the clues: `grep -o '[1-9]' /content/sudoku.txt | wc -l`.

</details>

✅ **Expected:** Your 9×9 puzzle printed with `?` in the empty cells, and the number of clues (typically 28-32).

In [40]:
%%bash
CODE=71265H # <- write YOUR code here
wget -q -O /content/sudoku.txt https://raw.githubusercontent.com/AntSanVar/DistributedSystemData/main/Session01/sudoku_$CODE.txt
ls -l /content/sudoku.txt
cat /content/sudoku.txt
echo "clues: $(grep -o '[1-9]' /content/sudoku.txt | wc -l)"

-rw-r--r-- 1 root root 162 Nov 24 15:34 /content/sudoku.txt
? 6 ? ? ? 4 ? ? ?
? ? ? ? 2 ? ? ? 7
? ? 3 6 5 ? 9 ? ?
7 ? ? 4 6 ? 2 5 ?
? ? 4 ? ? ? ? 7 ?
? ? 1 2 ? ? 3 4 ?
? ? ? 1 ? 2 7 9 ?
9 ? 7 ? ? 6 ? ? 1
? ? ? ? ? ? ? 6 5
clues: 29


## 4.2 Let Hadoop solve it

Run the `sudoku` program of the examples jar on **your file**. The solver prints the completed grid and how many solutions it found.

<details><summary>💡 Hint</summary>

`hadoop jar <examples jar> sudoku <your file>`. Use the full path of your file (for example `/content/sudoku_07.txt`).

</details>

✅ **Expected:** A completed 9×9 grid and the line `Found 1 solutions`.

In [42]:
%%bash
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.5.0.jar sudoku /content/sudoku.txt 2>&1 | tail -n 12


Solving /content/sudoku.txt
8 6 9 7 1 4 5 3 2 
1 4 5 3 2 9 6 8 7 
2 7 3 6 5 8 9 1 4 
7 3 8 4 6 1 2 5 9 
5 2 4 8 9 3 1 7 6 
6 9 1 2 7 5 3 4 8 
4 5 6 1 8 2 7 9 3 
9 8 7 5 3 6 4 2 1 
3 1 2 9 4 7 8 6 5 

Found 1 solutions


❓ **Question:** Write here the **solved Sudoku** that Hadoop found (9 lines of 9 digits, as in the output).

✍ *Your solution*:
```
8 6 9 7 1 4 5 3 2
1 4 5 3 2 9 6 8 7
2 7 3 6 5 8 9 1 4
7 3 8 4 6 1 2 5 9
5 2 4 8 9 3 1 7 6
6 9 1 2 7 5 3 4 8
4 5 6 1 8 2 7 9 3
9 8 7 5 3 6 4 2 1
3 1 2 9 4 7 8 6 5
```

❓ **Question:** How many solutions did it find? What would it mean if it found 0? Or more than 1?

✍ *Your answer*:
- **Number of solutions:** It found exactly **1** solution.
- **Meaning of 0:** A result of 0 solutions means that the Sudoku puzzle contains contradictory clues and is mathematically impossible to solve.
- **Meaning of > 1:** If it finds more than 1 solution, it means the puzzle is under-constrained (lacks sufficient clues to uniquely determine a single completed state).

## 4.3 Is it really distributed?

You have a running YARN. Ask it which applications have finished (`yarn application -list -appStates FINISHED`) and compare with what you saw after the `grep` job in step 3.10.

<details><summary>💡 Hint</summary>

If the Sudoku had run on the cluster it would appear as a new application with a name and an id.

</details>

✅ **Expected:** Only the two `grep` jobs (`grep-search`, `grep-sort`) are listed: **the Sudoku solver ran in a single JVM on the client**, it is not a MapReduce job.

In [44]:
%%bash
yarn application -list -appStates FINISHED 2>&1 | grep -E "application_|Total number" | cut -c1-110

Total number of applications (application-types: [], states: [FINISHED] and tags: []):2
application_1795532992554_0001	         grep-search	           MAPREDUCE	      root	root.default	          FIN
application_1795532992554_0002	           grep-sort	           MAPREDUCE	      root	root.default	          FIN


❓ **Question:** Does your Sudoku appear in the list? Is this program a MapReduce job? What does that tell you about the examples jar: is everything in it distributed?

✍ *Your answer*:
- **Does it appear:** No, the Sudoku application does not appear in the finished YARN applications list.
- **Is it MapReduce:** No, it is a standard sequential backtracking application that runs locally inside a single client JVM process.
- **What it tells us about the jar:** The examples JAR acts as a general-purpose playground. It contains both distributed MapReduce applications (like `wordcount` and `pi`) and standard non-distributed Java utilities (like `sudoku` and `pentomino`).

## 4.4 Bonus: how many solutions?

A good Sudoku has **exactly one** solution. Make a copy of your puzzle (`cp`), **delete a few more clues** replacing numbers by `?` (use `sed`, or edit the copy), and solve it again.


(Another idea: swap files with a classmate and solve theirs.)

<details><summary>💡 Hint</summary>

For example `sed '1s/[1-9]/?/' sudoku.txt > hard.txt` replaces the first digit of the first line by a `?`. Check the result with `cat`, then solve the new file.

</details>

✅ **Expected:** `Found N solutions` with N much bigger than 1 (the more clues you remove, the more solutions).

In [45]:
%%bash
cd /content
sed '1,3s/[1-9]/?/g' sudoku.txt > hard.txt
cat hard.txt
echo "----- SOLUTIONS -----"
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.5.0.jar sudoku /content/hard.txt 2>&1 | tail -n 3

? ? ? ? ? ? ? ? ?
? ? ? ? ? ? ? ? ?
? ? ? ? ? ? ? ? ?
7 ? ? 4 6 ? 2 5 ?
? ? 4 ? ? ? ? 7 ?
? ? 1 2 ? ? 3 4 ?
? ? ? 1 ? 2 7 9 ?
9 ? 7 ? ? 6 ? ? 1
? ? ? ? ? ? ? 6 5
----- SOLUTIONS -----
3 1 2 8 7 9 4 6 5 

Found 1917000 solutions


❓ **Question:** How many solutions does it find now? How many clues can you remove before the puzzle has more than one solution?

✍ *Your answer*:
- **How many solutions:** After replacing the first three rows with `?`, the solver found **1,917,000** solutions.
- **Removing clues:** In a standard 9x9 Sudoku, you can theoretically have a unique solution with as few as 17 clues. Removing even a single key clue from a minimal unique-solution puzzle will cause the number of solutions to increase to 2 or more.

# Part 5 – Clean up

## 5.1 Stop the daemons

Stop YARN and HDFS (`stop-yarn.sh`, `stop-dfs.sh`) and check with `jps` that nothing is left.

<details><summary>💡 Hint</summary>

Stop in the reverse order you started.

</details>

✅ **Expected:** Only `Jps` remains.

In [ ]:
%%bash
stop-yarn.sh
stop-dfs.sh
jps

### Final questions

Answer each one in the box below it.

❓ **Question:** Which daemon belongs to HDFS and which to YARN? Which one stores the blocks?

✍ *Your answer*:
- **HDFS Daemons:** `NameNode`, `DataNode`, and `SecondaryNameNode`.
- **YARN Daemons:** `ResourceManager` and `NodeManager`.
- **Daemon that stores the blocks:** The `DataNode` is responsible for physically storing blocks of data on its local disks.

❓ **Question:** What is the difference between standalone and pseudo-distributed mode? Why are the `grep` paths different?

✍ *Your answer*:
- **Standalone vs. Pseudo-distributed:** Standalone mode runs all components locally inside a single JVM, utilizing the local OS file system for storage. Pseudo-distributed mode runs separate Hadoop daemons (NameNode, DataNode, ResourceManager, etc.) as individual JVM processes communicating over the loopback network (`localhost`), utilizing HDFS for storage.
- **Why paths are different:** Standalone paths reference the local filesystem directory hierarchy (e.g. `./input`), whereas pseudo-distributed paths are virtual file paths pointing to locations inside the virtual HDFS filesystem managed by the NameNode (e.g. `/input`).

❓ **Question:** What would you need to change to turn this into a *real* cluster of several machines?

✍ *Your answer*:
To transition to a multi-node cluster, you must:
1. Configure SSH keys across all target machines so the master node can access worker nodes without passwords.
2. In `core-site.xml`, change `hdfs://localhost:9000` to point to the actual IP address or hostname of the master node.
3. In `yarn-site.xml`, configure `yarn.resourcemanager.hostname` to point to the master node IP/hostname.
4. Increase `dfs.replication` in `hdfs-site.xml` to an appropriate value (e.g., 3).
5. Populate the `workers` configuration file (formerly `slaves`) with the IP addresses or hostnames of all worker machines where DataNodes and NodeManagers should run.

❓ **Question:** What happens to your HDFS data when the Colab runtime is recycled? Is HDFS in Colab useful for real work?

✍ *Your answer*:
- **When recycled:** When the Colab runtime is recycled, the virtual machine instance is deleted, and all local directories (including `/tmp` and `/usr/local/hadoop` containing the HDFS storage) are permanently wiped.
- **Is HDFS in Colab useful:** No, HDFS on Colab is not suitable for production workloads or real storage tasks because of its ephemeral nature. However, it is highly useful as a free, zero-configuration sandbox for educational purposes, rapid prototyping, and debugging MapReduce logic.